In [54]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [55]:
data_files = ["crema_data_MFCC_20x301.npz", "crema_data_MFCC_40x301.npz", "crema_data_STFT_257x299.npz"]

In [56]:
def random_number():
    return int(str(time.time()).split(".")[1])

In [ ]:
for data_file in data_files:    
    print("\n")
    print('#' * 50)
    print(f"#{data_file:^{48}}#")    
    print('#' * 50)
    data = np.load(data_file) 
    
    # Extract features and labels
    X = data['features']      # Features matrix
    y = data['emotions']      # Emotion labels
    filenames = data['filenames']  # Filenames (if you saved them)
    
    print(f"Data loaded successfully!")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Unique emotions: {np.unique(y)}")
    print(f"\nEmotion distribution:")
    unique, counts = np.unique(y, return_counts=True)
    for emotion, count in zip(unique, counts):
        print(f"  {emotion}: {count} ({count/len(y)*100:.1f}%)")
    
    # Encode labels to numbers
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f"\nLabel mapping:")
    for emotion, code in zip(le.classes_, range(len(le.classes_))):
        print(f"  {emotion} -> {code}")
    
    # Split data, 90% train, 10% test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.1, random_state=random_number(), stratify=y_encoded
    )
    print(f"\nTrain size: {len(X_train)}")
    print(f"Test size: {len(X_test)}")
    
    # Scale features (important for many models)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=random_number()),
        'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
        'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
        'Decision Tree': DecisionTreeClassifier(random_state=random_number(), max_depth=10),
        'Random Forest (50)': RandomForestClassifier(n_estimators=50, random_state=random_number()),
        'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=random_number()),
        'Random Forest (200)': RandomForestClassifier(n_estimators=200, random_state=random_number()),
        'Random Forest (300)': RandomForestClassifier(n_estimators=200, random_state=random_number()),
        'Random Forest (400)': RandomForestClassifier(n_estimators=200, random_state=random_number()),
        'SVM (RBF)': SVC(kernel='rbf', random_state=random_number()),
        'SVM (Linear)': SVC(kernel='linear', random_state=random_number()),
        'LDA': LinearDiscriminantAnalysis(),
        'QDA': QuadraticDiscriminantAnalysis(),
    }
    
    # Train and evaluate
    print("\n")
    print('#' * 50)
    print(f"#{data_file:^{48}}#")
    print(f"#{'Training Result':^{48}}#")
    print('#' * 50)
    
    results = []
    
    for name, model in models.items():
        try:
            print(f"\nTraining {name}...", end=" ")
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            
            results.append({
                'Model': name,
                'Accuracy': f'{acc:.4f}',
                'Accuracy_Value': acc
            })
            print(f"✓ Accuracy: {acc:.4f}")
            
        except Exception as e:
            print(f"✗ Error: {str(e)[:60]}")
            results.append({
                'Model': name,
                'Accuracy': 'Error',
                'Accuracy_Value': 0
            })
    
    # Summary
    print('#' * 50)
    print(f"#{data_file:^{48}}#")
    print(f"#{'Summary':^{48}}#")
    print('#' * 50)
    results_df = pd.DataFrame(results)
    results_df_sorted = results_df.sort_values('Accuracy_Value', ascending=False)
    print(results_df_sorted[['Model', 'Accuracy']].to_string(index=False))
    
    # Best model
    best = results_df_sorted.iloc[0]
    print(f"\n🏆 BEST MODEL: {best['Model']} with accuracy {best['Accuracy']}")
    
    # Train best model again and show detailed results
    best_model_name = best['Model']
    best_model = models[best_model_name]
    best_model.fit(X_train_scaled, y_train)
    y_pred_best = best_model.predict(X_test_scaled)
    
    print("\n")
    print('#' * 100)
    print(f"#{data_file:^{98}}#")
    print(f"#{'Training Result':^{98}}#")
    print('#' * 100)
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred_best):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred_best, target_names=le.classes_))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_best)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{data_file}\nConfusion Matrix - {best_model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()



##################################################
#           crema_data_MFCC_20x301.npz           #
##################################################
Data loaded successfully!
X shape: (7440, 6020)
y shape: (7440,)
Unique emotions: ['ANG' 'DIS' 'FEA' 'HAP' 'NEU' 'SAD']

Emotion distribution:
  ANG: 1037 (13.9%)
  DIS: 648 (8.7%)
  FEA: 717 (9.6%)
  HAP: 393 (5.3%)
  NEU: 4208 (56.6%)
  SAD: 437 (5.9%)

Label mapping:
  ANG -> 0
  DIS -> 1
  FEA -> 2
  HAP -> 3
  NEU -> 4
  SAD -> 5

Train size: 6696
Test size: 744


##################################################
#           crema_data_MFCC_20x301.npz           #
#                Training Result                 #
##################################################

Training Logistic Regression... ✓ Accuracy: 0.5591

Training KNN (k=3)... ✓ Accuracy: 0.5739

Training KNN (k=5)... ✓ Accuracy: 0.5981

Training KNN (k=7)... ✓ Accuracy: 0.5995

Training Decision Tree... ✓ Accuracy: 0.5376

Training Random Forest (50)... 